# Lab 01 — Tabular Preprocessing with scikit-learn (NCA-GENL delta)

Skeleton notebook: setup and dataset cells **run as-is (fully offline)**; the parts that teach are `# TODO` cells you write yourself, guided by the exit criteria in `../lab-preprocessing-fundamentals.md`.

Fill the **results table at the bottom with your own measured numbers** — repo rule.

In [ ]:
# Environment check — runs as-is
import numpy as np, pandas as pd, matplotlib.pyplot as plt, sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
print("sklearn", sklearn.__version__)

In [ ]:
# Dataset — runs as-is. Wine (13 numeric features) + two synthetic categorical columns
# so we have both low- and high-cardinality categoricals to encode. Offline, seeded.
rng = np.random.default_rng(42)
df = load_wine(as_frame=True).frame.copy()
regions = np.array(["piemonte", "toscana", "veneto", "sicilia"])
df["region"] = regions[rng.integers(0, len(regions), len(df))]            # low cardinality (4)
df["vineyard_id"] = [f"V{n:03d}" for n in rng.integers(0, 60, len(df))]   # high cardinality (~58)
X, y = df.drop(columns="target"), df["target"]
num_cols = X.select_dtypes("number").columns.tolist()
cat_cols = ["region", "vineyard_id"]
df.head()

## 1. Scalers, side by side

`alcohol` (range ~11–15) and `proline` (range ~278–1680) live on wildly different scales — distance-based models will be dominated by `proline` unless we fix that.

In [ ]:
# Runs as-is: raw distributions
X[["alcohol", "proline"]].hist(bins=25, figsize=(8, 3)); plt.suptitle("raw"); plt.show()
X[["alcohol", "proline"]].describe().loc[["min", "max", "mean", "std"]]

In [ ]:
# TODO: apply MinMaxScaler and StandardScaler to the SAME two columns.
# Plot both transformed versions (reuse the .hist pattern above) and print min/max
# (MinMax) and mean/std (Standard) to confirm what each guarantees.
#
# Then answer in a markdown cell below, one sentence each:
#   - what each scaler does and when to prefer it
#   - one failure mode each (hint: add a fake outlier row and re-run MinMax)


*(your answers here)*

## 2. Encoders and the cardinality blowup

In [ ]:
# Runs as-is: one-hot on both categorical columns — count the output dimensions
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(X[cat_cols])
print("one-hot output dims:", sum(len(c) for c in ohe.categories_), "(from 2 input columns!)")
print("per column:", {col: len(cats) for col, cats in zip(cat_cols, ohe.categories_)})

In [ ]:
# TODO: encode `region` with OrdinalEncoder and explain in one sentence why using
# those integers in a linear model or KNN is wrong (fake ordering / fake distances),
# and when ordinal IS appropriate (true ordered categories; tree models tolerate it).
# Also: why does handle_unknown='ignore' matter at inference time? Demonstrate by
# transforming a row with region='puglia'.


## 3. The leakage demo (exam favorite)

Below is the **wrong** way, provided on purpose: the scaler is fit on train **and** test together before splitting the model's view of the world. Your job is to do it right and compare.

In [ ]:
# Runs as-is — the WRONG way (leakage): scaler sees the test distribution
X_tr, X_te, y_tr, y_te = train_test_split(X[num_cols], y, test_size=0.3, random_state=0, stratify=y)
scaler_leaky = StandardScaler().fit(pd.concat([X_tr, X_te]))   # <-- the sin
knn = KNeighborsClassifier(5).fit(scaler_leaky.transform(X_tr), y_tr)
print("leaky-fit test acc:", round(knn.score(scaler_leaky.transform(X_te), y_te), 4))

In [ ]:
# TODO: the RIGHT way. Build Pipeline([("scale", StandardScaler()), ("knn", ...)])
# and evaluate with cross_val_score(cv=5). The Pipeline re-fits the scaler on each
# training fold only.
#
# Then write the key sentence: why is the leaky version wrong even if the score
# barely moves here? (Small clean dataset => tiny leak. Name a situation where the
# gap gets large: distribution shift between train and test, tiny datasets,
# target leakage via preprocessing statistics.)


## 4. Tradeoffs, measured: scaling × model class

Four-way benchmark: {KNN, logistic regression, random forest} × {raw, scaled}. You already saw the punchline in section 1 — now produce the table yourself.

In [ ]:
# Runs as-is: the two preprocessors you'll plug models into
pre_scaled = ColumnTransformer([("num", StandardScaler(), num_cols),
                                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
pre_raw    = ColumnTransformer([("num", "passthrough", num_cols),
                                ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])

In [ ]:
# TODO: for each model in [KNeighborsClassifier(5), LogisticRegression(max_iter=2000),
# RandomForestClassifier(random_state=0)], run 5-fold cross_val_score with pre_raw
# and with pre_scaled. Collect results into a small DataFrame: rows=models,
# cols=[raw, scaled].
#
# Note: unscaled LogisticRegression will spam ConvergenceWarning — that is not a
# bug, it IS the lesson (gradient-based optimizer struggling on unscaled features).
# Mention it in your takeaway.


## Results (fill with YOUR numbers)

| model | raw CV acc | scaled CV acc | Δ |
|---|---|---|---|
| KNN (k=5) | | | |
| LogisticRegression | | | |
| RandomForest | | | |

**Takeaway sentence to be able to say cold:** *scaling changes distance- and gradient-based models; tree models don't care.*

### Exit criteria touched here
- [ ] purpose + pro + con for MinMaxScaler / StandardScaler / OneHotEncoder, no notes
- [ ] leakage via preprocessing + how Pipeline/ColumnTransformer prevents it
- [ ] my own measured numbers for scaling's effect per model class